# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore and process the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described using a Croissant schema accessible at the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Install the latest version of mlcroissant
!pip install -q mlcroissant

## 1. Data Loading
Load dataset metadata and establish a connection using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Access the metadata. Avoid subscripting metadata as per mlcroissant docs.
metadata_dict = dataset.metadata.to_json()
print("Dataset loaded:")
print(f"- Name: {metadata_dict['name']}")
print(f"- Description: {metadata_dict['description']}")

## 2. Data Overview
Review available record sets, fields, and their `@id` fields.

**Tip:** Use `dataset.record_sets` and `dataset.fields(record_set=<@id>)` to inspect structure. All references to dataset structure should use the `@id` for each entity.

In [ ]:
# List available record sets and their fields, referencing entities by their @id
record_sets = dataset.record_sets
print(f"Record sets found ({len(record_sets)}):\n")
for rset in record_sets:
    print(f"- RecordSet name: {rset['name']}  @id: {rset['@id']}")
    fields = list(dataset.fields(record_set=rset['@id']))
    print(f"  Fields ({len(fields)}):")
    for field in fields:
        fname = field.get('name', None)
        f_id = field.get('@id', None)
        ftype = field.get('dataType', None)
        print(f"    - {fname} (id: {f_id}) type: {ftype}")
    print("")

## 3. Data Extraction
Extract data from the record set(s) of interest into Pandas DataFrames for further analysis.

> For each extraction, the record set `@id` and field `@id`s are used. If multiple record sets are available, process them all.

In [ ]:
# Prepare a DataFrame per RecordSet
dataframes = {}
for rset in record_sets:
    record_set_id = rset['@id']
    print(f"Loading records for RecordSet: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  {len(df)} records loaded. Columns (field @id):\n  {df.columns.tolist()}\n")
    else:
        print("  No records in this RecordSet.\n")

## 4. Exploratory Data Analysis (EDA)
Apply EDA steps on a key record set. Select a numeric field (by its `@id`) and perform filtering, normalization, and aggregation.

- 1. Filter records by a numeric field (e.g., molecular_weight)
- 2. Normalize the field
- 3. Group by a relevant field (e.g., protein family, gene name, if available)

> Update the record set ID and field IDs below according to your dataset structure as reviewed above.

In [ ]:
# Pick the first non-empty record set
if dataframes:
    first_rs_id = next(iter(dataframes))
    df = dataframes[first_rs_id]
    print(f"Using RecordSet @id: {first_rs_id}")
    # Inspect numeric fields
    print("Numeric columns detected:")
    for col in df.columns:
        # crude numeric detection
        if pd.api.types.is_numeric_dtype(df[col]):
            print(f"- {col}")
    # Example: use 'cr:field/molecular_weight' if it exists, otherwise pick the first numeric
    numeric_candidates = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
        print(f"Using {numeric_field_id} as numeric field for analysis.\n")
        # Filter
        threshold = df[numeric_field_id].mean() # or another value
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered to records where {numeric_field_id} > {threshold:.2f} (mean):  {len(filtered_df)} records remain.")
        # Normalize
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(filtered_df[[numeric_field_id, norm_col]].head())
        # Try grouping by a categorical field (not numeric and not an index)
        cat_candidates = [c for c in df.columns if df[c].dtype=='object' and c != numeric_field_id]
        if cat_candidates:
            group_field = cat_candidates[0]
            print(f"\nGrouping by {group_field}:")
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(grouped_df.head())
    else:
        print("No numeric fields found to analyze.")
else:
    print("No available records for EDA.")

## 5. Visualization
Create visualizations of numeric field distributions and relationships.

> Note: If running in a non-interactive environment, ensure `matplotlib` is installed.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_candidates:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.tight_layout()
    plt.show()
    # If a group_field exists, boxplot
    if 'group_field' in locals():
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=group_field, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIR² dataset describing mass spectrometry analysis of extracellular vesicles from human mast cells, explored its record sets, performed initial EDA on numeric features referenced by their `@id`s, and visualized distributions. This pipeline can be adapted to further explore record set structure and relationships as needed for downstream analysis and ML workflows.